<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_04_2_early_stop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# T81-558: Applications of Deep Neural Networks

**Module 4: PyTorch Training**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).


# Module 4 Material

- Part 4.1: PyTorch Persistence [[Video]]() [[Notebook]](t81_558_class_04_1_pytorch_persist.ipynb)
- **Part 4.2: Early Stopping and Network Checkpointing** [[Video]]() [[Notebook]](t81_558_class_04_2_early_stop.ipynb)
- Part 4.3: K-Fold Cross-validation with PyTorch [[Video]]() [[Notebook]](t81_558_class_04_3_kfold.ipynb)
- Part 4.4: Training Schedules [[Video]]() [[Notebook]](t81_558_class_04_4_schedule.ipynb)
- Part 4.5: Regularization and Dropout [[Video]]() [[Notebook]](t81_558_class_04_5_dropout.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed. We also initialize the PyTorch device to either GPU/MPS (if available) or CPU.


In [1]:
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: using Google CoLab
Using device: cuda


# Part 4.2: Early Stopping

It can be difficult to determine how many epochs to cycle through to train a
neural network. Overfitting will occur if you train the neural network for too
many epochs, and the neural network will not perform well on new data, despite
attaining a good accuracy on the training set. Overfitting occurs when a neural
network is trained to the point that it begins to memorize rather than
generalize, as demonstrated in Figure 3.OVER.

One common technique for avoiding overfitting is early stopping. In early
stopping, the training process monitors the loss on a held-out validation set
at the end of each epoch. As long as the validation loss continues to decrease,
training proceeds. Once the validation loss stops improving — or begins to
increase — training is halted, optionally restoring the model weights from the
epoch where the best validation loss was observed.

Unlike frameworks such as Keras, PyTorch does not include early stopping as a
built-in feature. This is a deliberate reflection of PyTorch's design
philosophy. PyTorch is built around an explicit, imperative training loop that
the programmer writes and controls directly, rather than a high-level `fit`
method that manages training automatically. This gives PyTorch users fine-grained
control over every aspect of training, but it also means that conveniences like
early stopping, learning rate scheduling callbacks, and checkpoint management
must either be implemented by hand or sourced from a third-party library such
as PyTorch Lightning. The `EarlyStopping` class defined above is a
straightforward implementation of this pattern that integrates naturally into a
standard PyTorch training loop.

**Figure 3.OVER: Training vs. Validation Error for Overfitting**
![Training vs. Validation Error for Overfitting](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/class_3_training_val.png "Training vs. Validation Error for Overfitting")

It is important to segment the original dataset into several datasets:

- **Training Set**
- **Validation Set**
- **Holdout Set**

You can construct these sets in several different ways. The following programs demonstrate some of these.

The first method is a training and validation set. We train the neural network on the training data until the validation set stops improving. This attempts to stop at a near-optimal training point. This method will only give accurate "out of sample" predictions for the validation set; this is usually 20% of the data. The predictions for the training data will be overly optimistic, since we used it to train the neural network. Figure 3.VAL demonstrates how we divide the dataset.

**Figure 3.VAL: Training with a Validation Set**
![Training with a Validation Set](https://raw.githubusercontent.com/jeffheaton/t81_558_deep_learning/master/images/class_1_train_val.png "Training with a Validation Set")

Because PyTorch does not include a built-in early stopping function, we must define one ourselves. We will use the following **EarlyStopping** class throughout this course.

We can provide several parameters to the **EarlyStopping** object:

- **min_delta** This value should be kept small; it specifies the minimum change that should be considered an improvement. Setting it even smaller will likely have little impact.
- **patience** How long should the training wait for the validation error to improve?
- **restore_best_weights** You should usually set this to true, as it restores the weights to the values they were at when the validation set is the highest.

We will now see an example of this class in action.


In [2]:
import copy

class EarlyStopping:
    def __init__(
        self,
        patience: int = 5,
        min_delta: float = 0,
        restore_best_weights: bool = True
    ):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model: torch.nn.Module, val_loss: float) -> bool:
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

### Early Stopping with Classification

We will now see an example of classification training with early stopping. We will train the neural network until the error no longer improves on the validation set.


In [3]:
import time
import numpy as np
import pandas as pd
import torch
import tqdm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

def load_data(device: torch.device):
    df = pd.read_csv(
        "https://data.heatonresearch.com/data/t81-558/iris.csv",
        na_values=["NA", "?"]
    )
    le = LabelEncoder()
    x = df[["sepal_l", "sepal_w", "petal_l", "petal_w"]].values
    y = le.fit_transform(df["species"])
    species = le.classes_

    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.25, random_state=42
    )

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)

    x_train = torch.tensor(x_train, device=device, dtype=torch.float32)
    y_train = torch.tensor(y_train, device=device, dtype=torch.long)
    x_test = torch.tensor(x_test, device=device, dtype=torch.float32)
    y_test = torch.tensor(y_test, device=device, dtype=torch.long)

    return x_train, x_test, y_train, y_test, species

x_train, x_test, y_train, y_test, species = load_data(device)

BATCH_SIZE = 16
dataset_train = TensorDataset(x_train, y_train)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
dataset_test = TensorDataset(x_test, y_test)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

# Define model — no activation on final layer, CrossEntropyLoss handles softmax
model = nn.Sequential(
    nn.Linear(x_train.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, len(species)),
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
es = EarlyStopping()

epoch = 0
done = False
while epoch < 1000 and not done:
    epoch += 1
    steps = list(enumerate(dataloader_train))
    pbar = tqdm.tqdm(steps)
    model.train()
    for i, (x_batch, y_batch) in pbar:
        optimizer.zero_grad()
        y_batch_pred = model(x_batch)
        loss = loss_fn(y_batch_pred, y_batch)
        loss.backward()
        optimizer.step()
        loss_val = loss.item()

        if i == len(steps) - 1:
            model.eval()
            with torch.no_grad():
                pred = model(x_test)
                vloss = loss_fn(pred, y_test)
            if es(model, vloss):
                done = True
            pbar.set_description(
                f"Epoch: {epoch}, tloss: {loss_val:.4f}, "
                f"vloss: {vloss:.4f}, {es.status}"
            )
        else:
            pbar.set_description(f"Epoch: {epoch}, tloss: {loss_val:.4f}")

Epoch: 1, tloss: 0.6026, vloss: 0.5366, : 100%|██████████| 7/7 [00:00<00:00, 15.93it/s]
Epoch: 2, tloss: 0.3659, vloss: 0.2777, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 107.28it/s]
Epoch: 3, tloss: 0.1560, vloss: 0.1875, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 332.57it/s]
Epoch: 4, tloss: 0.0579, vloss: 0.1543, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 355.56it/s]
Epoch: 5, tloss: 0.1853, vloss: 0.0767, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 283.07it/s]
Epoch: 6, tloss: 0.1242, vloss: 0.0615, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 339.55it/s]
Epoch: 7, tloss: 0.0334, vloss: 0.0453, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 325.29it/s]
Epoch: 8, tloss: 0.0945, vloss: 0.0330, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 243.48it/s]
Epoch: 9, tloss: 0.0052, vloss

In [4]:
model.eval()
with torch.no_grad():
    pred = model(x_test)
    vloss = loss_fn(pred, y_test)

print(f"Loss = {vloss.item():.4f}")

Loss = 0.0090


As you can see from above, we did not use the total number of requested epochs. The neural network training stopped once the validation set no longer improved.


In [5]:
from sklearn.metrics import accuracy_score

model.eval()
with torch.no_grad():
    pred = model(x_test)

predicted_classes = torch.argmax(pred, dim=1)
correct = accuracy_score(y_test.cpu().numpy(), predicted_classes.cpu().numpy())
print(f"Accuracy: {correct:.4f}")

Accuracy: 1.0000


### Early Stopping with Regression

The following code demonstrates how we can apply early stopping to a regression problem. The technique is similar to the early stopping for the classification code that we just saw.

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read the Auto MPG dataset
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv",
    na_values=["NA", "?"]
)

# Handle missing value
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Select features and target
x = df[[
    "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "year", "origin"
]].values
y = df["mpg"].values

# Split into training and test sets
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Normalize features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Convert to PyTorch tensors
x_train = torch.tensor(x_train, device=device, dtype=torch.float32)
y_train = torch.tensor(y_train, device=device, dtype=torch.float32)
x_test = torch.tensor(x_test, device=device, dtype=torch.float32)
y_test = torch.tensor(y_test, device=device, dtype=torch.float32)

# Create DataLoaders
BATCH_SIZE = 16
dataset_train = TensorDataset(x_train, y_train)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
dataset_test = TensorDataset(x_test, y_test)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

# Define model
model = nn.Sequential(
    nn.Linear(x_train.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
).to(device)

# Loss function and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
es = EarlyStopping()

epoch = 0
done = False
while epoch < 1000 and not done:
    epoch += 1
    steps = list(enumerate(dataloader_train))
    pbar = tqdm.tqdm(steps)
    model.train()
    for i, (x_batch, y_batch) in pbar:
        optimizer.zero_grad()
        y_batch_pred = model(x_batch).flatten()
        loss = loss_fn(y_batch_pred, y_batch)
        loss.backward()
        optimizer.step()
        loss_val = loss.item()

        if i == len(steps) - 1:
            model.eval()
            with torch.no_grad():
                pred = model(x_test).flatten()
                vloss = loss_fn(pred, y_test)
            if es(model, vloss):
                done = True
            pbar.set_description(
                f"Epoch: {epoch}, tloss: {loss_val:.4f}, "
                f"vloss: {vloss:.4f}, EStop:[{es.status}]"
            )
        else:
            pbar.set_description(f"Epoch: {epoch}, tloss: {loss_val:.4f}")

Epoch: 1, tloss: 103.0915, vloss: 79.2384, EStop:[]: 100%|██████████| 19/19 [00:00<00:00, 89.34it/s]
Epoch: 2, tloss: 41.9121, vloss: 23.9704, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 320.81it/s]
Epoch: 3, tloss: 14.8819, vloss: 14.8574, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 326.26it/s]
Epoch: 4, tloss: 31.4927, vloss: 10.2772, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 315.72it/s]
Epoch: 5, tloss: 5.4470, vloss: 7.9848, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 322.24it/s]
Epoch: 6, tloss: 12.3442, vloss: 7.8017, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 327.32it/s]
Epoch: 7, tloss: 3.8572, vloss: 8.8285, EStop:[No improvement in the last 1 epochs]: 100%|██████████| 19/19 [00:00<00:00, 271.07it/s]
Epoch: 8, tloss: 2.8006, vloss: 9.7784, EStop:[No improvement in the last 2 ep

Finally, we evaluate the error.


In [7]:
model.eval()
with torch.no_grad():
    pred = model(x_test)

score = torch.sqrt(nn.functional.mse_loss(pred.flatten(), y_test))
print(f"Final score (RMSE): {score.item():.4f}")

Final score (RMSE): 2.4665


The code below sets up a neural network and reads the data (for predictions), but it does not clear the model directory or fit the neural network. The code loads the weights from the previous fit. Now we reload the network and perform another prediction. The RMSE should match the previous one exactly if we saved and reloaded the neural network correctly.
